In [0]:
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

In [0]:
import json

import pyspark.sql.functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns, safe_cast
from common_utils.scdfunctions import scd_type_1


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("customers_silver")


# ============================================================
# 1. READ CONFIG
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Reading Silver configuration from %s", config_path)

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 2. SOURCE / TARGET
# ============================================================

source_table = config["source"]["source_table"]
source_name = config["source"]["source_name"]
target_table = config["target"]["target_table"]


# ============================================================
# 3. CUSTOMER TRANSFORMATION
# ============================================================

def transform_customer(df, config):

    logger.info("[%s] Starting customer transformation", source_name)

    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    df = rename_columns(
        df,
        config.get("rename_columns", {})
    )


    # --------------------------------------------------------
    # Customer name
    # --------------------------------------------------------

    if "customer_name" in df.columns:

        logger.info("[%s] Transforming customer name", source_name)

        df = (
            df
            .withColumn("customer_name", F.lower(F.col("customer_name")))
            .withColumn("name_parts", F.split(F.col("customer_name"), ","))
            .withColumn("first_name", F.trim(F.expr("get(name_parts, 1)")))
            .withColumn("last_name", F.trim(F.expr("get(name_parts, 0)")))
        )


    # --------------------------------------------------------
    # City
    # --------------------------------------------------------

    if "city" in df.columns:

        logger.info("[%s] Standardising city", source_name)

        df = df.withColumn("city", F.upper(F.col("city")))


    # --------------------------------------------------------
    # Postcode
    # --------------------------------------------------------

    if "postcode" in df.columns:

        logger.info("[%s] Cleaning postcode", source_name)

        df = df.withColumn("postcode", F.regexp_replace(F.col("postcode"), r"\.0$", "")
        )


    # --------------------------------------------------------
    # Country
    # --------------------------------------------------------

    logger.info("[%s] Adding country", source_name)

    df = df.withColumn("country", F.lit("USA"))


    # --------------------------------------------------------
    # Drop columns
    # --------------------------------------------------------

    drop_columns = [
        c for c in config.get("drop_columns", [])
        if c in df.columns
    ]

    if drop_columns:

        logger.info(
            "[%s] Dropping columns: %s",
            source_name,
            drop_columns
        )

        df = df.drop(*drop_columns)


    # --------------------------------------------------------
    # Cast columns
    # --------------------------------------------------------

    df = safe_cast( df, config.get("cast_columns", {}) )


    # --------------------------------------------------------
    # Deduplication
    # --------------------------------------------------------

    dedup = config.get("deduplication", {})

    if dedup.get("enabled", False):

        logger.info("[%s] Removing duplicate records", source_name)

        key_columns = dedup["key_columns"]

        order_expressions = []

        for item in dedup["order_by"]:

            col = F.col(item["column"])

            if item.get("direction", "asc").lower() == "desc":

                col = (
                    col.desc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.desc_nulls_first()
                )

            else:

                col = (
                    col.asc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.asc_nulls_first()
                )

            order_expressions.append(col)

        window = (
            Window
            .partitionBy(*key_columns)
            .orderBy(*order_expressions)
        )

        df = (
            df
            .withColumn("rn", F.row_number().over(window))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )


    # --------------------------------------------------------
    # Final columns
    # --------------------------------------------------------

    logger.info("[%s] Selecting final Silver columns", source_name)

    df = df.select(
        *config.get("select_columns", [])
    )

    logger.info("[%s] Customer transformation completed", source_name)

    return df


# ============================================================
# 4. READ BRONZE
# ============================================================

logger.info(
    "[%s] Reading from Bronze table %s",
    source_name,
    source_table
)

df = spark.read.table(source_table)

logger.info(
    "[%s] Successfully read Bronze table",
    source_name
)


# ============================================================
# 5. TRANSFORM CUSTOMER
# ============================================================

cleaned_customer = transform_customer(
    df,
    config
)


# ============================================================
# 6. SCD / INITIAL LOAD
# ============================================================

if spark.catalog.tableExists(target_table):

    scd = config.get("scd", {})

    if scd.get("type") == 1:

        logger.info(
            "[%s] Starting SCD Type 1",
            source_name
        )

        scd_type_1(
            spark=spark,
            source_df=cleaned_customer,
            target_table=target_table,
            key_columns=scd.get("key_columns", [])
        )

    else:

        raise ValueError(
            f"Unsupported SCD type [{scd.get('type')}] "
            f"for source [{source_name}]"
        )

else:

    logger.info(
        "[%s] Target table does not exist - creating %s",
        source_name,
        target_table
    )

    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS
        retaildataplatform.silver
    """)

    (
        cleaned_customer
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    logger.info(
        "[%s] Silver table created successfully",
        source_name
    )


# ============================================================
# 7. COMPLETION
# ============================================================

logger.info(
    "[%s] Bronze to Silver pipeline completed successfully",
    source_name
)

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
def transform_column(df):
    df=df.withColumn('customer_name',F.lower(df['customer_name']))\
        .withColumn('name_parts',F.split(F.col('customer_name'),","))\
        .withColumn('Firs_name',F.expr("get(name_parts,1)"))\
        .withColumn('Last_name',F.expr("get(name_parts,0)"))\
        .withColumn('postcode', F.regexp_replace(F.col('postcode'), r'\.0$', ''))\
        .withColumn('city',F.upper(F.col('city')))\
        .withColumn('valid_from',F.from_unixtime('valid_from').cast('timestamp'))\
        .withColumn('valid_to',F.from_unixtime(F.expr("try_cast(valid_to as long)")).cast('timestamp'))\
        .withColumn('country',F.lit('USA'))

    df=df.drop('name_parts','customer_name','file_path')
    df=df.dropDuplicates(['customer_id'])

    return df.select(
    'customer_id',
    'Firs_name',
    'Last_name',
    'tax_id',
    'tax_code',
    'state',
    'city',
    'country',
    'postcode',
    'street',
    'number',
    'unit',
    'region',
    'district',
    'lon',
    'lat',
    'ship_to_address',
    'valid_from',
    'valid_to',
    'units_purchased',
    'loyalty_segment',
    'last_updated_ts',
    )

df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
cleaned_customer=transform_column(df)
cleaned_customer.display()

target_table="retaildataplatform.silver.customers"

if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print('Table Exists proceed with SCD-1 process.')
    # Load Target
    target=DeltaTable.forName(spark,target_table) # Use this when your Delta table is registered in the metastore (like Hive or Unity Catalog).
    # SCD Type-1
    # Match in source and target --> Key exist in target
    (
    target.alias("target").merge(cleaned_customer.alias('source'),"target.customer_id=source.customer_id")

    # Update-> When records are Matching
    .whenMatchedUpdateAll()

    # Insert-> When records are not Matching
    .whenNotMatchedInsertAll()

    .execute()
        
    )
else:
    print('Table does not exist- First create Schema(Table)')
    spark.sql("create schema if not exists retaildataplatform.silver")
    cleaned_customer.write.format('delta').saveAsTable('retaildataplatform.silver.customers')